In [ ]:
! echo $GOOGLE_APPLICATION_CREDENTIALS

In [ ]:
from typing import Dict, List, Union

from google.cloud import aiplatform
from google.protobuf import json_format
from google.protobuf.struct_pb2 import Value

import pandas as pd
import pickle
import os


def predict_custom_trained_model_sample(
    project: str,
    endpoint_id: str,
    instances: Union[Dict, List[Dict]],
    location: str = "us-east4",
    api_endpoint: str = "us-east4-aiplatform.googleapis.com",
):
    """
    `instances` can be either single instance of type dict or a list
    of instances.
    """
    # The AI Platform services require regional API endpoints.
    client_options = {"api_endpoint": api_endpoint}
    # Initialize client that will be used to create and send requests.
    # This client only needs to be created once, and can be reused for multiple requests.
    client = aiplatform.gapic.PredictionServiceClient(client_options=client_options)
    # The format of each instance should conform to the deployed model's prediction input schema.
    instances = instances if isinstance(instances, list) else [instances]
    instances = [
        json_format.ParseDict(instance_dict, Value()) for instance_dict in instances
    ]
    parameters_dict = {}
    parameters = json_format.ParseDict(parameters_dict, Value())
    endpoint = client.endpoint_path(
        project=project, location=location, endpoint=endpoint_id
    )
    response = client.predict(
        endpoint=endpoint, instances=instances, parameters=parameters
    )
    print("response")
    print(" deployed_model_id:", response.deployed_model_id)
    # The predictions are a google.protobuf.Value representation of the model's predictions.
    predictions = response.predictions
    for prediction in predictions:
        print(" prediction:", dict(prediction))

    return predictions

In [ ]:
df = pd.read_json("data/echantillon_test_llamandement.json")

In [ ]:
ENDPOINT_ID = "799252594416418816"
PROJECT_ID = "36002300583"

In [ ]:
def endpoint_predict_sample(
    project: str, location: str, instances: list, endpoint: str
):
    aiplatform.init(project=project, location=location)

    endpoint = aiplatform.Endpoint(endpoint)

    prediction = endpoint.predict(instances=instances)
    print(prediction)
    return prediction

In [ ]:
idx = 0
length = len(df)
instances = []
# TODO: Make calls one by one because batches return shuffled results
while idx < length:
    print(f"idx {idx}")
    prompt = SummaryPromptBuilder.build_prompt(df.iloc[idx]["Exposé amdt"])
    instances.append(
        {
            "inputs": prompt,
            "parameters": {"max_new_tokens": 128, "temperature": 1.0, "top_k": 1},
        }
    )
    idx += 1

results = endpoint_predict_sample(
    project=PROJECT_ID,
    endpoint=ENDPOINT_ID,
    instances=instances,
    location="us-east4",
)

display(results)

In [ ]:
# Define the path to the data folder
DATA_FOLDER = os.getenv("DATA_FOLDER", "data")

# Define the path to the pickle file
pickle_file = os.path.join(DATA_FOLDER, "query_llamandement.pickle")

# Pickle the result variable
with open(pickle_file, "wb") as f:
    pickle.dump(results, f)

In [ ]:
# Assuming 'pickle_file' is the path to your pickle file
with open(pickle_file, "rb") as f:
    prediction_obj = pickle.load(f)
prediction_obj

In [ ]:
# Initialize a list to hold modified predictions
modified_predictions = []

# Define the prefixes to remove
prefixes_to_remove = [
    "Le sujet de l'amendement est de ",
    "Le sujet de l'amendement est d'",
]

# Iterate over the predictions and remove the prefixes
for prediction in prediction_obj.predictions:
    modified_prediction = prediction
    for prefix in prefixes_to_remove:
        if prediction.startswith(prefix):
            modified_prediction = prediction[len(prefix) :]
            break  # Stop checking other prefixes once one is removed
    modified_prediction = modified_prediction.capitalize()
    modified_predictions.append(modified_prediction)

# Uppercase the first character of each prediction in modified_predictions
modified_predictions = [prediction.capitalize() for prediction in modified_predictions]

In [ ]:
# Load the CSV file into a DataFrame
previous_sample_df = pd.read_csv("data/pinned_échantillons_objets.csv")

previous_sample_df.rename(
    columns={"Objet (Exp LLaMa 3)": "Objet (LLaMa 3 8B)"}, inplace=True
)
previous_sample_df.drop(columns=["Objet (LLaMa 3)"], inplace=True)
previous_sample_df["Objet (LLaMandement 13B)"] = modified_predictions
display(previous_sample_df)
# for idx, pred in enumerate(prediction_obj.predictions):
#     print(f"idx {idx}")
#     print(pred)

In [ ]:
previous_sample_df.to_csv(
    "data/pinned_échantillons_objets_llamandement.csv",
    index=False,
    encoding="utf-8-sig",
)